# Lab 2: Custom Common-Envelope (CE) Step

When running **POSYDON**, the core–envelope boundary can only be defined using three fixed hydrogen mass fraction thresholds:  
$$
X_H = 0.01,\; 0.1,\; \text{or}\; 0.3
$$

### Motivation
---
In some cases, however, we may want to explore **other values** of the hydrogen mass fraction to define the core boundary.  
This gives us more flexibility in studying how different assumptions affect the outcome of common-envelope evolution.

### Goal of this Lab
---
In this lab, we will:
1. Develop a **custom CE step** that allows us to use **any arbitrary value** of the hydrogen mass fraction to determine the core–envelope boundary.  
2. Introduce and use our functions (`calculate_binding_energy` and `calculate_E_lambda_CE`) to calculate the post-CE outcome.  
3. Apply this custom CE step to a **population of double neutron stars (DNS)**.  

For this exercise, begin by downloading a **population of DNS systems**.  
We will then **re-run this population** using our custom CE step and compare the resulting DNS distributions when different definitions of the core boundary are used.  


We don't want to edit the source code so we will need to make our own step. The Lab folder there is a python file custom_CE_step.py that will substitute the existing CE_step in the POSYDON flow. We will edit the CE and complete the pieces that are missing. We are mostly going to edit only these three function: `calculate_binding_energy`, `calculate_lamda_CE` and `CEE_simple_alpha_prescription`


### Step 1

The function `CEE_simple_alpha_prescription` takes as inputs the `donor` star and the `comp_star`. These are object of the class Single_star. 
In our first step we need to define the variables of pre CE masses,radii and the states of the two stars as `m1_i`, `radius1`,`state1_i` for the donor and `m2_i`, `radius2`,`state2_i`. These values are atrributes of the object Single star. Look at the file `POSYDON/posydon/binary_evol/singlestar.py` of how these attributes are defined. 

Hint: You can find them in `STARPROPERTIES`

<details>
<summary>Click for the solution</summary>

```python
    m1_i = donor.mass
    radius1 = 10**donor.log_R
    state1_i = donor.state
    m2_i = comp_star.mass
    radius2 = 10**comp_star.log_R
```

### Step 2
We want to define variables related to the binary period and the alpha_CE that has been given from ini the file. Look in the binary_star.py how to access the binary orbital period from the binary Object. Look at the initialization function of class for the alpha_CE. 

<details>
<summary>Click for the solution</summary>

```python
    period_i = binary.orbital_period
    alpha_CE = self.common_envelope_efficiency 
```

### Step 3 

The final profile of the stars is can be access from the attribute single.profile. The outcome will be an long array as we saw in Lab 1. Since we built our function to handle pandas DataFrames. Load the profiles in to two variable `donor_prof` and `comp_prof` as pandas DataFrames.

<details>
<summary>Click for the solution</summary>

```python
    donor_prof = pd.DataFrame(donor.profile)
    comp_prof = pd.DataFrame(comp_star.profile)
```

### Step 4 

Take the two function from Lab 1 and paste them under the function in the .py file. We also need our `calculate_lamdbda_CE` function to return two more variables related to the donor's core and donor's radius as found from our arbitrary boundary condition. Make the necessary changes in the `calculate_lamdbda_CE` to return the mass and radius of the core from the star's profile.

<details>
<summary>Click for the solution</summary>

```python
    def calculate_binding_energy(self,core_definition_H_fraction,star,common_envelope_alpha_thermal = 1):

        Grav_energy = 0.0
        U_i = 0.0

        radius = np.array(star.radius)
        internal_energy = np.array(star.energy)
        mass = np.array(star.mass) 
        print()
        dm = np.concatenate((-1 * np.diff(mass),[mass[-1]]))

        zones = star[star.x_mass_fraction_H > core_definition_H_fraction].index
        zone_core = zones[-1]
        for i in range(zone_core):
            Grav_energy_i = (-const.standard_cgrav * mass[i]
                                   * const.Msun * dm[i]*const.Msun
                                   / (radius[i]*const.Rsun))
            # integral of gravitational energy as we go deeper into the star
            Grav_energy = Grav_energy + Grav_energy_i
            U_i = U_i + internal_energy[i]*dm[i]*const.Msun

        # binding energy of the enevelope equals its gravitational energy +
        # an a_th fraction of its internal energy
        Ebind_i = Grav_energy + common_envelope_alpha_thermal * U_i
        return Ebind_i
    
    
    
    def calculate_lambda_CE(self,core_definition_H_fraction,star,common_envelope_alpha_thermal = 1):
        zones = star[star.x_mass_fraction_H > core_definition_H_fraction].index
        zone_i = zones[-1]
        E_bind = self.calculate_binding_energy(core_definition_H_fraction,star,common_envelope_alpha_thermal = 1)
        mass = star.mass
        radius = star.radius
        M_donor = mass[0]
        M_core = mass[zone_i]
        M_envelope = M_donor - M_core
        R_core = radius[zone_i]
        R = radius[0]
        lambda_CE = - const.standard_cgrav *(M_donor*const.Msun)*(M_envelope*const.Msun)/(R*const.Rsun*E_bind)
        return lambda_CE,M_core,R_core
    ```

### Step 5
 Calculate the initial seperation of the binary before it underwent the CE. We only have access to the binary's period but not the seperation through the binary object. Thankfully you don't need to write a function, in posydon/utils there is a file that contains a lot of useful functions that are being used throughout the code. Look into the `common_functions.py` for a function that calculates the orbital seperation from period, what units is it in ? Use that function and calculate the pre-CE seperation in cgs units.

<details>
<summary>Click for the solution</summary>

```python
     separation_i = const.Rsun * cf.orbital_separation_from_period(
            period_i, m1_i, m2_i)   # in cgs units
    ```

### Step 6 


Now we have all the parameters need to calculate the orbital energy that was required to eject the envelope. 

<details>
<summary>Click for the solution</summary>

```python
        eorb_i = (-0.5 * const.standard_cgrav * m1_i * const.Msun
                  * m2_i * const.Msun / separation_i)
        

        eorb_postCEE = eorb_i + ebind_i/alpha_CE

        separation_postCEE = (-0.5 * const.standard_cgrav * mc1_i * const.Msun
                              * mc2_i * const.Msun / eorb_postCEE)
```

Now we are ready to use our step and we need to introduce to the flow. We will do that by editing the .ini file. Copy the ini file from posydon/popsyn 

In step_CE it gives us a option through the argument absolute import `absolute_import =` to point to our file with our own step. Edit the absolute_import to contain the path to our file and the class name of our CE_step. 

<details>
<summary>Click for the solution</summary>

```python
    absolute_import = ['/home/kasdaglie/blue/kasdaglie/testing_sc_stuff/custom_CE_step.py','StepCEE']
```

Before we start our run we also need to change the `interpolation_method` to the `nearest_neighbour` in all the steps that have that option. The reason behind is.......

### Re-evolving a population
---

To test if everything is running fine we can try and re-run a subsection of the DNS population in our notebooks for debbuging. Go to the ini file and change the number of binaries to 10 

In [ ]:

import os 
os.environ['PATH_TO_POSYDON_DATA'] = '/your/path/PATH_TO_POSYDON_DATA'
os.environ['PATH_TO_POSYDON'] = '/your/path//POSYDON/'

from posydon.popsyn.synthetic_population import BinaryPopulation
from posydon.popsyn.io import binarypop_kwargs_from_ini
from posydon.utils.common_functions import convert_metallicity_to_string
import argparse
ini_kw = binarypop_kwargs_from_ini('./population_params.ini')
ini_kw['metallicity'] = 1.0
ini_kw['file_name'] = './solar_DNS.h5'
#ini_kw['common_envelope_efficiency'] = 10
poprun = BinaryPopulation(**ini_kw)


In [ ]:
poprun.evolve(from_hdf=True,tqdm=True)
poprun.save('./test_solar_DNS.h5' )

If no errors occured during your run, inspect your population to make sure that everything looks okay. 

Now let's run our entire population:
 - Choose a value for the boundary core and alpha from the spreadsheet and replace these values in your ini file.
 - Change the number of binaries from 10 to 1800. Go to the `run_job.sh` file and replace the path for the export of the `PATH_TO_POSYDON` and `PATH_TO_POSYDON_DATA` with your own path.  
 - Make sure you also put you email in the `--mail-user` option. 
 - Submit your job to the cluster by running the command ```sbatch run_job.sh```


The job will take 10-15 minutes. While we wait for our job to run we will continue with the rest of the Lab to calculate the merger times of the solar type DNS in the original population and later 

### Calculate the merger times of DNS
The inspiral time of double Neutron stars can be estimated by: 
___

$$
    \tau_{merger} = 10^7 yr \, P_{orb,h}^{8/3} \, \big ( \frac{M}{M_\odot} \big )^{-2/3} \big ( \frac{\mu}{M_\odot} \big )^{-1} (1 - e^2)^{7/2}
$$


We will calculate the merger times from the fiducial population and then compare with the merger times from the population with our custom CE step. 



### Step 1 
Import the population using pandas arrays and load the history. 

<details>
<summary>Solution</summary>

```python 
    pop= pd.read_hdf("./solar_DNS.h5", key = 'history')


```




If you look at the history of the first binary by choosing `pop.loc[0]` after the event of `CC2` it's the step where the DNS is newly form. We want to extract the information at this point of the binary evolution and not the final step as the binary's orbital period is evolved as a double compact object. 


### Step 2 

Extract the binary properties for the newly formed DNS from the history. We want the line after the event of `CC2`we can do that by using `np.where()` which will return the `iloc` indexes where `event == 'CC2'` and index after that one will be only the lines containg the newly formed DNS. We also need to make sure that all of the systems end up being NS so should to a final check on the state of both stars. 


<details>
<summary> Hint </summary>
   The `iloc` indexes containg the systems we want can be found by: 

```python 
    np.where(pop.event == 'CC2')[0] + 1
```



<details>
<summary>Solution</summary>

```python 
    newly_formed_systems = pop.iloc[np.where(pop.event.values == "CC2")[0]+1]
    formed_DNS = newly_formed_systems[(newly_formed_systems.S1_state == 'NS') & (newly_formed_systems.S2_state == 'NS')]
```

### Step 3 
Instead of writing our function for the merger time, **POSYDON** already includes a function that calculates the inspiral time of contact objects. Look into the file `common_function.py` and find a function that will calculate the inspiral time for a given period, eccentricity and masses. 

<details>
<summary>Solution</summary>

```python 
    from posydon.utils.common_functions import inspiral_timescale_from_orbital_period
```

### Step 4
Look at the function `inspiral_timescale_from_orbital_period` we need to iterate along the `formed_DNS` an calculate the `spiral_time`. The merger time would be the time of the newly formed DNS plus the `spiral_time`.  

<details>
<summary>Solution</summary>

```python 
    t_inspiral = []
    for i in range(len(formed_DNS)):
        t = inspiral_timescale_from_orbital_period(
            formed_DNS.S1_mass.iloc[i],
            formed_DNS.S2_mass.iloc[i],
            formed_DNS.orbital_period.iloc[i],
            formed_DNS.eccentricity.iloc[i]
        )
        t_inspiral.append(t)

    t_inspiral = np.array(t_inspiral)
    t_mergers = t_inspiral + np.array(formed_DNS.time)/1e6
```

### Step 5

Combine the above steps into a function that would take a Dataframe population and will the merger times of only DNS. 

<details>
<summary>Solution</summary>

```python 
    def DNS_merger_times(pop):
        newly_formed_systems = pop.iloc[np.where(pop.event.values == "CC2")[0]+1]
        formed_DNS = newly_formed_systems[(newly_formed_systems.S1_state == 'NS') & (newly_formed_systems.S2_state == 'NS')]
        t_inspiral = []
        for i in range(len(formed_DNS)):
            t = inspiral_timescale_from_orbital_period(
                formed_DNS.S1_mass.iloc[i],
                formed_DNS.S2_mass.iloc[i],
                formed_DNS.orbital_period.iloc[i],
                formed_DNS.eccentricity.iloc[i]
            )
            t_inspiral.append(t)

        t_inspiral = np.array(t_inspiral)
        t_inspiral = np.array(t_inspiral)
        t_mergers = t_inspiral + np.array(formed_DNS.time)/1e6
        return t_mergers
```


---
Hopefully by now your population has finished its run 
___ 
### Step 6 
    
Use the code bellow to plot a histogram of the pre-run DNS population. Overplot the merger times from your population. How do they compare ? Discuss with your rest of your group 


In [ ]:
#Histogram of the mergers 
t_mergers = DNS_merger_times(pop)
Hubble_time_Myr = 1.38e4 
plt.figure(figsize=(8,6))
plt.hist(np.log10(t_mergers), bins=50, color="steelblue", edgecolor="black",alpha = 0.4)
plt.xlabel(r"$\log_{10}(t_{\rm merger} \; [\mathrm{Myr}])$")
plt.ylabel("# of NS mergers")
plt.title("DNS Merger Times")
plt.axvline(np.log10(Hubble_time_Myr), color="red", linestyle="--", linewidth=2,
            label="Hubble time (13.8 Gyr)")
plt.show()